## Exploring the csv

In [11]:
import polars as pl

from utils import ROOT, wide_print
from pathlib import Path

In [12]:
dataset_name = Path(ROOT, 'data/data-designs-with-events_preflabel.csv')

In [13]:
events = pl.read_csv(dataset_name)

In [14]:
wide_print(events.head())

shape: (5, 3)
┌──────────────────────────────────┬────────────┬───────────────────────────────────────────────────────────────────┐
│ altTitle                         ┆ var1Label  ┆ tijdstipVarLabel                                                  │
│ ---                              ┆ ---        ┆ ---                                                               │
│ str                              ┆ str        ┆ str                                                               │
╞══════════════════════════════════╪════════════╪═══════════════════════════════════════════════════════════════════╡
│ AANVULVERGOEDPERSOONMNDBEDRAGBUS ┆ Rinpersoon ┆ Einde geldigheidsperiode attributen                               │
│ AANVULVERGOEDPERSOONMNDBEDRAGBUS ┆ Rinpersoon ┆ Aanvang geldigheidsperiode attributen                             │
│ ADOPTIEKINDEREN                  ┆ Rinpersoon ┆ Datum ingang familierechtelijke betrekking moeder van adoptiekind │
│ ADOPTIEKINDEREN                  ┆ Rinpe

In [15]:
events.describe()

statistic,altTitle,var1Label,tijdstipVarLabel
str,str,str,str
"""count""","""818""","""818""","""818"""
"""null_count""","""0""","""0""","""0"""
"""mean""",null,null,null
"""std""",null,null,null
"""min""","""AANVULVERGOEDPERSOONMNDBEDRAGB…","""Rinpersoon""","""AANVANGHBAAN"""
"""25%""",null,null,null
"""50%""",null,null,null
"""75%""",null,null,null
"""max""","""ZWPERSOONMNDBEDRAGBUS""","""Rinpersoon""","""sleutel EBB"""


In [17]:
events.shape

(818, 3)

In [20]:
events.head()

altTitle,var1Label,tijdstipVarLabel
str,str,str
"""AANVULVERGOEDPERSOONMNDBEDRAGB…","""Rinpersoon""","""Einde geldigheidsperiode attri…"
"""AANVULVERGOEDPERSOONMNDBEDRAGB…","""Rinpersoon""","""Aanvang geldigheidsperiode att…"
"""ADOPTIEKINDEREN""","""Rinpersoon""","""Datum ingang familierechtelijk…"
"""ADOPTIEKINDEREN""","""Rinpersoon""","""Datum ingang familierechtelijk…"
"""ADOPTIEKINDEREN""","""Rinpersoon""","""Jaar van adoptie/vestiging"""


In [18]:
n_datasets = events["altTitle"].n_unique()
n_event_variables = events["tijdstipVarLabel"].n_unique()
print(f"We have {n_datasets} datasets and {n_event_variables} unique event variables.")
# NOTE: duplicated rows?! - 516 seems to correspond to what I got in the later query? this means that the same tijdstipVarLabel occurs in multiple datasets(!)
# numbers fluctuate? inner join/missing?

We have 258 datasets and 516 unique event variables.


Some events seem repeated every year, such as birth date of parents:

In [19]:
wide_print(events.filter(pl.col("altTitle").str.contains("PIVA")))

shape: (15, 3)
┌─────────────────────────┬────────────┬─────────────────────────────────────────────────────────────────┐
│ altTitle                ┆ var1Label  ┆ tijdstipVarLabel                                                │
│ ---                     ┆ ---        ┆ ---                                                             │
│ str                     ┆ str        ┆ str                                                             │
╞═════════════════════════╪════════════╪═════════════════════════════════════════════════════════════════╡
│ PIVAADRESOBJECTBUS      ┆ Rinpersoon ┆ Datum aanvang adresbewoning persoon                             │
│ PIVAADRESOBJECTBUS      ┆ Rinpersoon ┆ Datum beeïndiging adresbewoning persoon                         │
│ PIVABURGERLIJKESTAATBUS ┆ Rinpersoon ┆ Datum waarop een burgerlijke staat van de persoon is ingegaan.  │
│ PIVABURGERLIJKESTAATBUS ┆ Rinpersoon ┆ Datum waarop een burgerlijke staat van de persoon is beëindigd. │
│ PIVAMIGRATIEBUS     

Notes
- income data??

In [7]:
# inpatab -> from tax declaration, spolisbus -> from companies
wide_print(events.filter(pl.col("altTitle") == "SPOLISBUS"))

shape: (5, 3)
┌───────────┬────────────┬────────────────────────────────────────────────────────────────────────────────┐
│ altTitle  ┆ var1Label  ┆ tijdstipVarLabel                                                               │
│ ---       ┆ ---        ┆ ---                                                                            │
│ str       ┆ str        ┆ str                                                                            │
╞═══════════╪════════════╪════════════════════════════════════════════════════════════════════════════════╡
│ SPOLISBUS ┆ Rinpersoon ┆ Datum aanvang inkomstenverhouding of aanvang verslagperiode                    │
│ SPOLISBUS ┆ Rinpersoon ┆ Datum aanvang inkomstenopgave                                                  │
│ SPOLISBUS ┆ Rinpersoon ┆ Datum einde inkomstenopgave                                                    │
│ SPOLISBUS ┆ Rinpersoon ┆ De einddatum van de inkomstenverhouding of einde verslagperiode                │
│ SPOLISBUS ┆ 

### Questions
- where are the descriptives of the variables? not sure it's on the portal? https://w3id.org/odissei/cv/cbs/variableThesaurus/c721a0a69c98684bc4b968faafc4508a8e9a28dbe4347b4a978edc696341548cc
- how do we define "unique" events from the data designs? 

### Notes

Trying to understand the [KG](https://kg.odissei.nl/odissei/-/stories/ODISSEI-Knowledge-Graph-the-story) and relation to the metadata
- Data designs. "Each individual record in the ODISSEI portal represents an individual data design." 
- There are 258 distinct `altTitle`. Angelica mentioned the same number of data designs. So, is `altTitle` equivalent to data design? 
  - from the KG: a dataset can have multiple "data designs". A data design can have multiple DOIs.
  - what is "alternative name"?


In [8]:
events["altTitle"].n_unique()

258

variables with tijdstip = "time" columns in any dataset


In [9]:
events["altTitle"].unique()

altTitle
str
"""POLISHOOFDBAANBUS"""
"""GBAHUISHOUDENSBUS"""
"""WUSUITKERING1ATAB"""
"""GBANATIONALITEITBUS"""
"""SECMPENSIOENMNDBEDRAGBUS"""
…
"""ANWMNDBEDRAGBUS"""
"""INDICWLZTAB"""
"""DIPLOMAMBOTAB"""


In [10]:
events.shape

(818, 3)

### Idea
- harmonize the events over time but not their content
- is it correct that events defined by `altTitle` and `tijdstipVarLabel` define "same events" over time?
   - `altTitle` can refer to different columns in different datasets in different years?
- what do we do with events that do not have a `tijdstip`, such as INPA? are these events also harmonized in the KG?
